# 08 · 模型怎么知道库存是 37 件？
客户问「商品 P28 还有多少件？」库存随时会变，模型需要向仓库查询。这节课用两条虚构库存代替仓库数据库。

**今天先做第 1 步。** 点击下面的代码格，按 Shift + Enter。第一格只需要 Python；后面用本机内核「LLM 工具课（本机）」。

## 1 · 人直接查库存
先猜：如果把参数换成 A17，结果会怎样？

In [11]:
stock = {"P28": 37, "A17": 0}

def get_stock(sku):
    return stock.get(sku)  # 没有这条记录时返回 None

get_stock("A7")

**先停在这里。** 37 来自字典。这个函数就是工具最简单的样子；目前还没有调用模型，也没有用到 MCP。可以把最后一行改成 A17、Z99 分别试试：零库存与查不到有何区别？

## 2 · 用普通函数询问：库存服务有哪些工具？
先看 `available = list_tools()`。它启动同目录的库存服务，发出 MCP 请求，等到工具清单再返回。连接细节放在 [sync_inventory.py](sync_inventory.py)。这里不会查库存，也不会调用模型。

In [12]:
from pathlib import Path
import sys
if Path("08-tools-mcp/sync_inventory.py").is_file():
    sys.path.insert(0, str(Path("08-tools-mcp").resolve()))

from sync_inventory import list_tools, call_tool

available = list_tools()
print(available)

[{'name': 'get_stock', 'description': 'Look up current stock by product code (SKU). Unknown does not mean zero.', 'inputSchema': {'properties': {'sku': {'title': 'Sku', 'type': 'string'}}, 'required': ['sku'], 'title': 'get_stockArguments', 'type': 'object'}, 'outputSchema': {'properties': {'result': {'title': 'Result', 'type': 'string'}}, 'required': ['result'], 'title': 'get_stockOutput', 'type': 'object'}}]


/Users/yorkchen/文稿 - 陈尧的MacBook Pro/VibeCoding/llm-from-zero/.venv/lib/python3.12/site-packages/pydantic_settings/sources/utils.py:51: IncompleteFieldDefinitionWarning: Field 'lifespan' has an incomplete definition: its annotation contains an unresolved forward reference, so settings sources may fail to correctly resolve its value. Call `model_rebuild()` on the model where the field is defined, once all the referenced types are defined.
  warnings.warn(  # noqa: B028  (called from many sources at differing depths, so no stacklevel is correct)
Processing request of type ListToolsRequest


## 3 · 模型第一次回答：写出想调用的函数
复用上节课的 generate 封装，读取 Mac 已有的 Qwen3-0.6B。传入的只有工具说明和对话，库存表不在提示词里。找找模型生成的函数名和参数。

In [13]:
from demo import load_generator
from huggingface_hub import snapshot_download

path = snapshot_download("Qwen/Qwen3-0.6B", local_files_only=True)
generate = load_generator(path)
messages = [
    {"role": "system", "content": "你是库存助手。查询库存必须调用工具，不要猜测。收到工具结果后直接用中文简短回答。其他问题直接回答。"},
    {"role": "user", "content": "请查一下商品 P28 现在有多少件库存？"},
]
# 把服务端的工具说明整理成模型需要的格式
tools = []
for item in available:
    function = {"name": item["name"], "description": item["description"],
                "parameters": item["inputSchema"]}
    tools.append({"type": "function", "function": function})

raw = generate(messages, tools)
print(raw)

device: mps


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


tokens: input=219 output=22
<tool_call>
{"name": "get_stock", "arguments": {"sku": "P28"}}
</tool_call>


## 4 · 谁真正去查？
模型刚才只产生文字。宿主程序读出调用请求，检查名字和参数，再通过 MCP 执行。看看 37 在这次模型与工具的交互中第一次出现在哪里。

In [14]:
import json, re

match = re.fullmatch(r"<tool_call>\s*(.+?)\s*</tool_call>", raw, re.S)
if not match:
    raise ValueError("这次模型没有输出单个完整调用，请检查上格原文。")
call = json.loads(match[1])
args = call["arguments"]
if (call["name"] != "get_stock" or not isinstance(args, dict) or set(args) != {"sku"}
        or not isinstance(args["sku"], str) or not re.fullmatch(r"[A-Za-z0-9-]{1,32}", args["sku"])):
    raise ValueError("工具名或参数不符合本课约定。")

receipt = call_tool(call["name"], args)
print(receipt)

P28：37 件


/Users/yorkchen/文稿 - 陈尧的MacBook Pro/VibeCoding/llm-from-zero/.venv/lib/python3.12/site-packages/pydantic_settings/sources/utils.py:51: IncompleteFieldDefinitionWarning: Field 'lifespan' has an incomplete definition: its annotation contains an unresolved forward reference, so settings sources may fail to correctly resolve its value. Call `model_rebuild()` on the model where the field is defined, once all the referenced types are defined.
  warnings.warn(  # noqa: B028  (called from many sources at differing depths, so no stacklevel is correct)
Processing request of type CallToolRequest


## 5 · 模型第二次回答：把查询结果告诉用户
对比这次与第 3 步：模型没有换，权重也没变；输入里多了调用请求和一条 role=tool 的回执。

In [15]:
history = messages + [
    {"role": "assistant", "content": "", "tool_calls": [{"type": "function", "function": call}]},
    {"role": "tool", "content": receipt},
]
print(generate(history, tools))

tokens: input=262 output=11
P28：37 件库存。


回看：**Tool** 是查库存的函数；**function calling** 是模型写出函数名和参数；**MCP** 让程序能发现并执行另一个进程提供的工具。